Model training and comparison

Business objective: predict the maximum market value in EUR that a football player will reach during his career.

In this notebook I train three regression models and compare them with MAE, RMSE, and R2. The models are saved in the models folder so the project pipeline can load them later.

Premier point important: this is a regression problem, not a classification problem, because the target is a continuous value in euros. Accuracy or F1 would not be correct here. MAE, RMSE, and R2 are better aligned with the objective.

In [1]:
from pathlib import Path
import sys

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR if (CURRENT_DIR / "src").exists() else CURRENT_DIR.parent
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

from data import load_dataset_split


The target has a very skewed distribution. Most players have relatively low peak market values, while a small number of elite players reach extremely high values. Because of that, the models are trained with a log transformed target and predictions are converted back to euros.

In [2]:
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

X_train, X_test, y_train, y_test = load_dataset_split()

print("Training rows", X_train.shape[0])
print("Test rows", X_test.shape[0])
print("Feature count", X_train.shape[1])
print("Target median", round(float(y_train.median()), 2))
print("Target mean", round(float(y_train.mean()), 2))
print("Target max", round(float(y_train.max()), 2))


Training rows 31380
Test rows 7846
Feature count 14
Target median 800000.0
Target mean 3495277.41
Target max 200000000.0


The three models are chosen to make the comparison meaningful. Decision Tree is the simple baseline because it is easy to explain. Random Forest tests whether averaging many trees improves results. XGBoost is a stronger boosted tree model and should often perform well on tabular data.

In [3]:
def log_target_model(regressor):
    return TransformedTargetRegressor(
        regressor=regressor,
        func=np.log1p,
        inverse_func=np.expm1,
        check_inverse=False,
    )


models = {
    "decision_tree": log_target_model(
        DecisionTreeRegressor(
            max_depth=10,
            min_samples_leaf=20,
            random_state=42,
        )
    ),
    "random_forest": log_target_model(
        RandomForestRegressor(
            n_estimators=140,
            max_depth=14,
            min_samples_leaf=3,
            random_state=42,
            n_jobs=-1,
        )
    ),
    "xgboost_regressor": log_target_model(
        XGBRegressor(
            objective="reg:squarederror",
            n_estimators=350,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.9,
            colsample_bytree=0.9,
            reg_lambda=1.0,
            random_state=42,
            n_jobs=-1,
            tree_method="hist",
        )
    ),
}


MAE is the easiest metric to explain because it is the average error in euros. RMSE is useful because it punishes very large mistakes more strongly, which matters for expensive players. R2 gives a general view of how much variation the model explains.

In [4]:
def regression_metrics(y_true, y_pred):
    clean_predictions = np.maximum(np.asarray(y_pred, dtype=float), 0)
    return {
        "mae_eur": float(mean_absolute_error(y_true, clean_predictions)),
        "rmse_eur": float(np.sqrt(mean_squared_error(y_true, clean_predictions))),
        "r2": float(r2_score(y_true, clean_predictions)),
    }


rows = []

for model_key, model in models.items():
    print("Training", model_key)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    row = {"model_key": model_key}
    row.update(regression_metrics(y_test, predictions))
    rows.append(row)
    joblib.dump(model, MODELS_DIR / f"{model_key}.joblib")

comparison = pd.DataFrame(rows).sort_values("mae_eur")
comparison.to_csv(RESULTS_DIR / "model_training_comparison.csv", index=False)

print(comparison.to_string(index=False))


Training decision_tree
Training random_forest


Training xgboost_regressor


        model_key      mae_eur     rmse_eur       r2
xgboost_regressor 1.789689e+06 5.233763e+06 0.726058
    random_forest 1.893084e+06 5.733880e+06 0.671203
    decision_tree 2.205368e+06 6.731849e+06 0.546791


Decision rule for later: the final model should mainly be selected by the lowest MAE and RMSE, then checked with R2. If two models are close, the simpler or more explainable model is easier to defend in class.